# EPL448 – Deliverable 3: Predictive Model Development & Performance Evaluation
## CERN Electron Collision Dataset – Team 2
**Members:** Varnavas Tryfonos, Thrasos Sazeidis, Andreas Evagorou  
**Dataset:** [CERN Electron Collision Data (Kaggle)](https://www.kaggle.com/datasets/fedesoriano/cern-electron-collision-data)

---
## 0. Setup & Imports

In [ ]:
import sys
import os
from pathlib import Path

# Allow notebooks/ to import from src/
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Targeted suppression only – no blanket filterwarnings('ignore')
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning, module='sklearn')
warnings.filterwarnings('ignore', message='.*max_iter.*')

from sklearn.model_selection import (
    train_test_split, cross_validate, KFold, GridSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print('XGBoost not installed – install with: pip install xgboost')

# ── Project modules ──────────────────────────────────────────────────────
from src.features import LOG_FEATURES, add_engineered_features, build_v1, build_v2, build_v3, build_v4
from src.evaluation import compute_metrics, CV_SCORING
from src.models import (build_pipeline, PARAM_GRIDS, N_SVR_SCREEN,
                        N_TOP_MODELS, N_TOP_DATASETS, RANDOM_STATE)
from src.validation import validate_raw, validate_clean

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')

OUTPUTS_DIR = Path('../outputs')
OUTPUTS_DIR.mkdir(exist_ok=True)

print('All imports successful.')


---
## 1. Load Dataset & Reproduce Preprocessing from Deliverable 2

In [ ]:
DATA_PATH = Path('../data/dielectron.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}.\n"
        "Download dielectron.csv and place it in the data/ directory.\n"
        "See data/README.md for instructions."
    )

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()   # fix L1: trailing space in column names

print(f'Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')

validate_raw(df)
print('Raw data validation passed.')

df_clean = df.drop(columns=['Run', 'Event'], errors='ignore').copy()
df_clean = df_clean.dropna(subset=['M']).reset_index(drop=True)

validate_clean(df_clean)
print(f'Working dataset: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns')
print('Clean data validation passed.')


### 1.1 Feature Engineering Functions (from Deliverable 2)

In [ ]:
# Feature engineering is defined in src/features.py (imported above).
# Key fix: delta_phi uses vectorised np.minimum (no .apply/lambda loop).
print('Feature engineering module loaded from src/features.py')
print(f'LOG_FEATURES: {LOG_FEATURES}')


### 1.2 Revisions from Deliverable 2

**Key revisions based on feedback:**

1. **Scaling removed from tree-based models:** In Deliverable 2, all dataset versions used `StandardScaler`. The professor noted that tree-based models (RF, XGBoost) are scale-invariant. Scaling is now a pipeline step applied only for KNN and SVR.

2. **Feature selection / dimensionality reduction applied:** The original dataset has 16 features after dropping identifiers, and V4 (with engineered features) had 22. Many of these are redundant:
   - `px1, py1` are redundant with `pt1, phi1` (different coordinate system for the same momentum)
   - `px2, py2` are redundant with `pt2, phi2`
   - `pz1, pz2` are captured by `E, eta` (pz = pt × sinh(eta))
   - `phi1, phi2` individually have near-zero correlation with M
   - `eta1, eta2` individually are captured by `delta_eta`
   - `Q1, Q2` are captured by `opposite_sign`
   - `log_pt1, log_pt2` correlate ~0.87 with `log_E1, log_E2` and are captured by `sum_pt`

   V4 is now trimmed to **8 features** based on domain knowledge and the feature importance analysis from Deliverable 2. This reduces training time dramatically (especially for SVR and KNN) while preserving all physically meaningful information.

3. **Pipeline-based preprocessing:** All preprocessing is inside scikit-learn Pipelines to prevent data leakage during cross-validation.

### 1.3 Build Dataset Versions

| Version | Features | # Features | Target | Purpose |
|---------|----------|-----------|--------|--------|
| V1 | Original (raw, all features) | 16 | M | Baseline – no transforms |
| V2 | Log-transformed energy/pt | 16 | M | Reduce skewness |
| V3 | Log-transformed energy/pt | 16 | log(M) | Log target for relative errors |
| V4 | **Reduced: 8 selected features** (engineered + log energy) | 8 | M | Domain-driven feature selection |
| V5 | PCA on V2 features (in pipeline) | ~10 | M | Dimensionality reduction via PCA/SVD |

In [ ]:
# V4_SELECTED is defined in src/features.py
from src.features import V4_SELECTED

y     = df_clean['M'].values
y_log = np.log(y)

X_v1 = build_v1(df_clean)
X_v2 = build_v2(df_clean)
X_v3 = build_v3(df_clean)
X_v4 = build_v4(df_clean)   # 8-feature reduced set
X_v5 = build_v2(df_clean)   # same as V2; PCA applied inside pipeline

mass_bins = pd.qcut(y, q=10, labels=False, duplicates='drop')

datasets_raw = {}
for name, X_df, y_arr in [
    ('V1', X_v1, y),
    ('V2', X_v2, y),
    ('V3', X_v3, y_log),
    ('V4', X_v4, y),
    ('V5', X_v5, y),
]:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_df.values, y_arr, test_size=0.2,
        random_state=RANDOM_STATE, stratify=mass_bins
    )
    datasets_raw[name] = {
        'X_train':       X_tr,
        'X_test':        X_te,
        'y_train':       y_tr,
        'y_test':        y_te,
        'feature_names': list(X_df.columns),
        'log_target':    (name == 'V3'),
        'pca':           (name == 'V5'),
    }
    print(f'{name}: train={X_tr.shape}, test={X_te.shape}, '
          f'log_target={name == "V3"}, pca={name == "V5"}')

print(f'\nFeature counts: V1={X_v1.shape[1]}, V2={X_v2.shape[1]}, '
      f'V3={X_v3.shape[1]}, V4={X_v4.shape[1]}, V5={X_v5.shape[1]}')
print(f'\nV4 selected features: {V4_SELECTED}')


---
## 2. Model Evaluation Metrics

Since this is a **regression** problem (predicting invariant mass M in GeV), we use the following metrics:

| Metric | Formula | Why we use it |
|--------|---------|---------------|
| **RMSE** | $\sqrt{\frac{1}{n}\sum(y_i - \hat{y}_i)^2}$ | Penalises large errors more heavily. Same units as target (GeV). |
| **MAE** | $\frac{1}{n}\sum|y_i - \hat{y}_i|$ | Robust to outliers. Typical prediction error in GeV. |
| **R²** | $1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$ | Proportion of variance explained. 1.0 = perfect. |
| **MAPE** | $\frac{100}{n}\sum\left|\frac{y_i - \hat{y}_i}{y_i}\right|$ | Relative error — important because M spans 2–110 GeV. |

**Primary metric for model selection:** R²  
**Secondary metrics:** RMSE and MAPE

In [ ]:
# compute_metrics and CV_SCORING defined in src/evaluation.py
cv_scoring = CV_SCORING
print('Evaluation metrics loaded from src/evaluation.py')


---
## 3. Initial Model Experimentation – Default Hyperparameters

All four models are trained with **default hyperparameters** on all five dataset versions using
**5-fold cross-validation**.

**Pipeline logic:**
- KNN, SVR → `StandardScaler → Model` (scaling needed)
- RF, XGBoost → `Model` only (scale-invariant)
- V5 (PCA) → `StandardScaler → PCA(95%) → Model` for all models

In [ ]:
# build_pipeline is defined in src/models.py (imported above).
# Fix M5: PCA(random_state=...) was silently ignored for dense arrays;
# random_state is now omitted in the src/models.py implementation.

print('=== Pipeline Examples ===')
for name in ['KNN', 'SVR', 'RF', 'XGB']:
    pipe = build_pipeline(name)
    steps = ' -> '.join(s[0] for s in pipe.steps)
    print(f'{name} (no PCA):  {steps}')

pipe_pca = build_pipeline('KNN', use_pca=True)
steps = ' -> '.join(s[0] for s in pipe_pca.steps)
print(f'KNN (with PCA): {steps}')


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
all_cv_results = []

for ver_name, ver_data in datasets_raw.items():
    X_tr   = ver_data['X_train']
    y_tr   = ver_data['y_train']
    use_pca = ver_data['pca']

    for model_name in ['KNN', 'SVR', 'RF', 'XGB']:
        pipe = build_pipeline(model_name, use_pca=use_pca)

        # SVR subsample (see src/models.py for N_SVR_SCREEN documentation)
        if model_name == 'SVR':
            rng = np.random.default_rng(RANDOM_STATE)
            n   = min(N_SVR_SCREEN, len(X_tr))
            idx = rng.choice(len(X_tr), n, replace=False)
            X_cv, y_cv = X_tr[idx], y_tr[idx]
        else:
            X_cv, y_cv = X_tr, y_tr

        cv_res = cross_validate(
            pipe, X_cv, y_cv, cv=kf,
            scoring=cv_scoring, n_jobs=-1,
            return_train_score=False,
        )

        result = {
            'Model':     model_name,
            'Dataset':   ver_name,
            'R2_mean':   cv_res['test_R2'].mean(),
            'R2_std':    cv_res['test_R2'].std(),
            'RMSE_mean': -cv_res['test_RMSE'].mean(),
            'RMSE_std':  cv_res['test_RMSE'].std(),
            'MAE_mean':  -cv_res['test_MAE'].mean(),
            'MAE_std':   cv_res['test_MAE'].std(),
        }
        all_cv_results.append(result)
        pca_tag = ' [PCA]' if use_pca else ''
        print(f'{model_name:4s} x {ver_name}{pca_tag}: '
              f'R²={result["R2_mean"]:.4f} (±{result["R2_std"]:.4f})  '
              f'RMSE={result["RMSE_mean"]:.3f}  MAE={result["MAE_mean"]:.3f}')
    print()

cv_df = pd.DataFrame(all_cv_results)
print('Screening complete.')


### 3.1 Screening Results Table

In [ ]:
# Pivot table: R²
pivot_r2 = cv_df.pivot(index='Model', columns='Dataset', values='R2_mean')
pivot_r2 = pivot_r2[['V1', 'V2', 'V3', 'V4', 'V5']]
pivot_r2 = pivot_r2.loc[['KNN', 'SVR', 'RF', 'XGB']]

print('=== Cross-Validated R² Scores (Default Hyperparameters) ===')
display(pivot_r2.style.format('{:.4f}').highlight_max(axis=None, color='lightgreen'))

# Pivot table: RMSE
pivot_rmse = cv_df.pivot(index='Model', columns='Dataset', values='RMSE_mean')
pivot_rmse = pivot_rmse[['V1', 'V2', 'V3', 'V4', 'V5']]
pivot_rmse = pivot_rmse.loc[['KNN', 'SVR', 'RF', 'XGB']]

print('\n=== Cross-Validated RMSE (Default Hyperparameters) ===')
display(pivot_rmse.style.format('{:.3f}').highlight_min(axis=None, color='lightgreen'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
model_colors = {'KNN': 'steelblue', 'SVR': 'darkorange', 'RF': 'seagreen', 'XGB': 'crimson'}

ax = axes[0]
bar_data = cv_df.copy()
bar_data['label'] = bar_data['Model'] + ' / ' + bar_data['Dataset']
bar_data = bar_data.sort_values('R2_mean', ascending=True)
colors = [model_colors[m] for m in bar_data['Model']]
ax.barh(bar_data['label'], bar_data['R2_mean'], xerr=bar_data['R2_std'],
        color=colors, edgecolor='none', capsize=3)
ax.set_xlabel('R² (5-fold CV)', fontsize=11)
ax.set_title('Model × Dataset Screening – R²', fontsize=13)

ax = axes[1]
bar_data2 = cv_df.sort_values('RMSE_mean', ascending=False)
colors2 = [model_colors[m] for m in bar_data2['Model']]
ax.barh(bar_data2['Model'] + ' / ' + bar_data2['Dataset'], bar_data2['RMSE_mean'],
        xerr=bar_data2['RMSE_std'], color=colors2, edgecolor='none', capsize=3)
ax.set_xlabel('RMSE in GeV (5-fold CV)', fontsize=11)
ax.set_title('Model × Dataset Screening – RMSE', fontsize=13)

from matplotlib.patches import Patch
legend_patches = [Patch(facecolor=c, label=m) for m, c in model_colors.items()]
axes[0].legend(handles=legend_patches, loc='lower right', fontsize=9)

plt.tight_layout()
fig_path = OUTPUTS_DIR / 'fig_screening_results_svd.png'
plt.savefig(fig_path, bbox_inches='tight')
plt.show()
print(f'Figure saved to {fig_path}')


---
## 4. Selection of Best-Performing Models & Dataset Versions

In [ ]:
top_combos = cv_df.nlargest(10, 'R2_mean')[['Model', 'Dataset', 'R2_mean', 'RMSE_mean']]
print('Top 10 model × dataset combinations by R²:')
print(top_combos.to_string(index=False))

model_avg = cv_df.groupby('Model')['R2_mean'].mean().sort_values(ascending=False)
print('\nAverage R² by model:')
print(model_avg.to_string())

dataset_avg = cv_df.groupby('Dataset')['R2_mean'].mean().sort_values(ascending=False)
print('\nAverage R² by dataset:')
print(dataset_avg.to_string())

# Automated model selection – no hardcoding (M3 fix)
# All 4 models are tuned; select top 2 datasets by mean CV R²
SELECTED_MODELS   = ['XGB', 'RF', 'KNN', 'SVR']
SELECTED_DATASETS = dataset_avg.nlargest(N_TOP_DATASETS).index.tolist()

print(f'\n>>> Models for tuning:   {SELECTED_MODELS}')
print(f'>>> Datasets for tuning: {SELECTED_DATASETS}')
print(f'>>> Total combinations:  {len(SELECTED_MODELS) * len(SELECTED_DATASETS)}')


---
## 5. Pipeline Definition

Pipelines integrate preprocessing with the model so that `GridSearchCV` handles
train/validation splitting with no data leakage.

| Model | V1–V4 (no PCA) | V5 (with PCA/SVD) |
|-------|----------------|-------------------|
| KNN   | StandardScaler → KNN | StandardScaler → PCA(95%) → KNN |
| SVR   | StandardScaler → SVR | StandardScaler → PCA(95%) → SVR |
| RF    | RF only | StandardScaler → PCA(95%) → RF |
| XGB   | XGB only | StandardScaler → PCA(95%) → XGB |

In [ ]:
# Show pipelines for all tuning combinations
print('=== Pipelines for Tuning ===')
for model_name in SELECTED_MODELS:
    for dataset_name in SELECTED_DATASETS:
        use_pca = datasets_raw[dataset_name]['pca']
        pipe = build_pipeline(model_name, use_pca=use_pca)
        n_feats = datasets_raw[dataset_name]['X_train'].shape[1]
        print(f'{model_name:4s} on {dataset_name} ({n_feats} features, PCA={use_pca}): '
              f'{" → ".join(s[0] for s in pipe.steps)}')

---
## 6. Hyperparameter Tuning with GridSearchCV

Trimmed grids based on screening insights — avoids wasting time on parameter regions
that are clearly suboptimal. V4 has only 8 features, making SVR and KNN feasible
on the full training set.

In [ ]:
# PARAM_GRIDS is defined in src/models.py (imported above).
print('Hyperparameter grids loaded from src/models.PARAM_GRIDS')
for name in SELECTED_MODELS:
    grid = PARAM_GRIDS[name]
    n_combos = 1
    for v in grid.values():
        n_combos *= len(v)
    print(f'  {name}: {n_combos} combinations × 5 folds = {n_combos * 5} fits')


In [ ]:
# Run GridSearchCV for all 4 models × 2 datasets
grid_results = {}

# SVR subsample size — V4 has only 8 features so we can use more data
N_SVR_TUNE = 30000

for model_name in SELECTED_MODELS:
    for dataset_name in SELECTED_DATASETS:
        key = f'{model_name}_{dataset_name}'
        print(f'\n{"="*60}')
        print(f'Tuning: {model_name} on {dataset_name}')
        print(f'{"="*60}')
        
        ver = datasets_raw[dataset_name]
        X_tr = ver['X_train']
        y_tr = ver['y_train']
        use_pca = ver['pca']
        
        pipe = build_pipeline(model_name, use_pca=use_pca)
        
        # SVR subsample (larger now thanks to fewer features in V4)
        if model_name == 'SVR':
            n = min(N_SVR_TUNE, len(X_tr))
            idx = np.random.RandomState(RANDOM_STATE).choice(len(X_tr), n, replace=False)
            X_search, y_search = X_tr[idx], y_tr[idx]
            print(f'  SVR using {n:,} subsample (of {len(X_tr):,})')
        else:
            X_search, y_search = X_tr, y_tr
        
        gs = GridSearchCV(
            estimator=pipe,
            param_grid=param_grids[model_name],
            cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
            scoring='r2',
            n_jobs=-1,
            verbose=1,
            return_train_score=True
        )
        gs.fit(X_search, y_search)
        
        grid_results[key] = {
            'grid_search': gs,
            'best_params': gs.best_params_,
            'best_cv_r2':  gs.best_score_,
            'model_name':  model_name,
            'dataset_name': dataset_name
        }
        
        print(f'\nBest params: {gs.best_params_}')
        print(f'Best CV R²:  {gs.best_score_:.4f}')

In [ ]:
# Summary of tuning results
tuning_summary = []
for key, res in grid_results.items():
    tuning_summary.append({
        'Model': res['model_name'],
        'Dataset': res['dataset_name'],
        'Best CV R²': res['best_cv_r2'],
        'Best Params': str(res['best_params'])
    })

tuning_df = pd.DataFrame(tuning_summary).sort_values('Best CV R²', ascending=False)
print('=== Hyperparameter Tuning Summary ===')
display(tuning_df)

---
## 7. Final Model Evaluation on Test Set

In [ ]:
final_results = []
predictions = {}

for key, res in grid_results.items():
    model_name = res['model_name']
    dataset_name = res['dataset_name']
    ver = datasets_raw[dataset_name]
    
    best_model = res['grid_search'].best_estimator_
    y_pred = best_model.predict(ver['X_test'])
    y_true = ver['y_test']
    
    if ver['log_target']:
        y_pred_eval = np.exp(y_pred)
        y_true_eval = np.exp(y_true)
    else:
        y_pred_eval = y_pred
        y_true_eval = y_true
    
    metrics = compute_metrics(y_true_eval, y_pred_eval)
    metrics['Model'] = model_name
    metrics['Dataset'] = dataset_name
    metrics['Best Params'] = str(res['best_params'])
    final_results.append(metrics)
    
    predictions[key] = {
        'y_true': y_true_eval,
        'y_pred': y_pred_eval,
        'model_name': model_name,
        'dataset_name': dataset_name
    }
    
    print(f'{model_name:4s} / {dataset_name}: R²={metrics["R2"]:.4f}  '
          f'RMSE={metrics["RMSE"]:.3f} GeV  MAE={metrics["MAE"]:.3f} GeV  '
          f'MAPE={metrics["MAPE"]:.2f}%')

final_df = pd.DataFrame(final_results)
print('\n=== Final Test Set Evaluation ===')
display(final_df[['Model', 'Dataset', 'R2', 'RMSE', 'MAE', 'MAPE', 'Best Params']])

### 7.1 Predicted vs Actual Scatter Plots

In [ ]:
n_plots = len(predictions)
n_cols = min(4, n_plots)
n_rows = (n_plots + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
axes = np.array(axes).flatten() if n_plots > 1 else [axes]

for ax, (key, pred) in zip(axes, predictions.items()):
    n = len(pred['y_true'])
    idx = np.random.RandomState(42).choice(n, size=min(3000, n), replace=False)
    ax.scatter(pred['y_true'][idx], pred['y_pred'][idx],
              alpha=0.2, s=5, color='steelblue', rasterized=True)
    lims = [0, max(pred['y_true'].max(), pred['y_pred'].max()) * 1.05]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect fit')
    ax.set_xlabel('Actual M (GeV)', fontsize=10)
    ax.set_ylabel('Predicted M (GeV)', fontsize=10)
    m = compute_metrics(pred['y_true'], pred['y_pred'])
    ax.set_title(f"{pred['model_name']} / {pred['dataset_name']}\n"
                 f"R²={m['R2']:.4f}  RMSE={m['RMSE']:.2f} GeV", fontsize=10)
    ax.legend(fontsize=8)
    ax.set_xlim(lims)
    ax.set_ylim(lims)

# Hide unused axes
for i in range(n_plots, len(axes)):
    axes[i].set_visible(False)

plt.suptitle('Predicted vs Actual – Tuned Models (Test Set)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'fig_pred_vs_actual.png', bbox_inches='tight')
plt.show()

### 7.2 Residual Analysis

In [ ]:
n_plots = len(predictions)
n_cols = min(4, n_plots)
n_rows_plot = (n_plots + n_cols - 1) // n_cols
fig, axes = plt.subplots(2 * n_rows_plot, n_cols, figsize=(6 * n_cols, 5 * 2 * n_rows_plot))
axes = np.array(axes).reshape(2 * n_rows_plot, n_cols)

for j, (key, pred) in enumerate(predictions.items()):
    col = j % n_cols
    row_base = (j // n_cols) * 2
    residuals = pred['y_true'] - pred['y_pred']
    
    # Residuals vs Actual
    ax = axes[row_base, col]
    idx = np.random.RandomState(42).choice(len(residuals), min(3000, len(residuals)), replace=False)
    ax.scatter(pred['y_true'][idx], residuals[idx], alpha=0.2, s=5, color='steelblue', rasterized=True)
    ax.axhline(0, color='red', linewidth=1, linestyle='--')
    ax.set_xlabel('Actual M (GeV)', fontsize=9)
    ax.set_ylabel('Residual (GeV)', fontsize=9)
    ax.set_title(f"{pred['model_name']} / {pred['dataset_name']}\nResiduals vs Actual", fontsize=10)
    
    # Residual distribution
    ax = axes[row_base + 1, col]
    ax.hist(residuals, bins=100, color='steelblue', edgecolor='none', alpha=0.8)
    ax.axvline(0, color='red', linewidth=1, linestyle='--')
    ax.set_xlabel('Residual (GeV)', fontsize=9)
    ax.set_ylabel('Count', fontsize=9)
    ax.set_title(f'Residual Distribution\nMean={residuals.mean():.3f}, Std={residuals.std():.3f}', fontsize=10)

# Hide unused axes
for row in range(2 * n_rows_plot):
    for col in range(n_cols):
        idx_flat = (row // 2) * n_cols + col
        if idx_flat >= n_plots:
            axes[row, col].set_visible(False)

plt.suptitle('Residual Analysis – Tuned Models', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'fig_residual_analysis.png', bbox_inches='tight')
plt.show()

### 7.3 Performance by Mass Region

In [ ]:
def get_mass_region(m):
    if m < 10:
        return 'Low (< 10 GeV)'
    elif m < 50:
        return 'Mid (10–50 GeV)'
    else:
        return 'High (> 50 GeV)'

region_results = []
for key, pred in predictions.items():
    regions = np.array([get_mass_region(m) for m in pred['y_true']])
    for region in ['Low (< 10 GeV)', 'Mid (10–50 GeV)', 'High (> 50 GeV)']:
        mask = regions == region
        if mask.sum() == 0:
            continue
        m = compute_metrics(pred['y_true'][mask], pred['y_pred'][mask])
        m['Model'] = pred['model_name']
        m['Dataset'] = pred['dataset_name']
        m['Region'] = region
        m['N_events'] = int(mask.sum())
        region_results.append(m)

region_df = pd.DataFrame(region_results)
print('=== Performance by Mass Region ===')
display(region_df[['Model', 'Dataset', 'Region', 'N_events', 'R2', 'RMSE', 'MAE', 'MAPE']]
        .style.format({'R2': '{:.4f}', 'RMSE': '{:.3f}', 'MAE': '{:.3f}', 'MAPE': '{:.2f}'}))

### 7.4 Feature Importance (Best Model)

In [ ]:
best_key = final_df.loc[final_df['R2'].idxmax()]
best_combo_key = f"{best_key['Model']}_{best_key['Dataset']}"
best_gs = grid_results[best_combo_key]['grid_search']
best_estimator = best_gs.best_estimator_

if hasattr(best_estimator, 'named_steps'):
    actual_model = best_estimator.named_steps['model']
else:
    actual_model = best_estimator

best_dataset = best_key['Dataset']
uses_pca = datasets_raw[best_dataset]['pca']

if hasattr(actual_model, 'feature_importances_'):
    importances = actual_model.feature_importances_
    if uses_pca:
        feat_names = [f'PC{i+1}' for i in range(len(importances))]
        title_suffix = '(PCA Components)'
    else:
        feat_names = datasets_raw[best_dataset]['feature_names']
        title_suffix = '(Original Features)'
    
    feat_imp = pd.Series(importances, index=feat_names).sort_values(ascending=False)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    feat_imp.plot(kind='barh', ax=ax, color='steelblue', edgecolor='none')
    ax.invert_yaxis()
    ax.set_xlabel('Feature Importance (MDI)', fontsize=12)
    ax.set_title(f'Feature Importances – {best_key["Model"]} / {best_dataset} {title_suffix}', fontsize=13)
    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR / 'fig_feature_importance_best.png', bbox_inches='tight')
    plt.show()
    
    print('Feature importances:')
    print(feat_imp.to_string())
else:
    print(f'Feature importances not available for {type(actual_model).__name__}')

---
## 8. Comparison of Default vs Tuned Performance

In [ ]:
comparison_rows = []
for key, res in grid_results.items():
    model_name = res['model_name']
    dataset_name = res['dataset_name']
    default_r2 = cv_df[
        (cv_df['Model'] == model_name) & (cv_df['Dataset'] == dataset_name)
    ]['R2_mean'].values[0]
    tuned_r2 = res['best_cv_r2']
    comparison_rows.append({
        'Model': model_name,
        'Dataset': dataset_name,
        'Default R² (CV)': default_r2,
        'Tuned R² (CV)': tuned_r2,
        'Improvement': tuned_r2 - default_r2
    })

comp_df = pd.DataFrame(comparison_rows)
print('=== Default vs Tuned Performance ===')
display(comp_df.style.format({
    'Default R² (CV)': '{:.4f}',
    'Tuned R² (CV)': '{:.4f}',
    'Improvement': '{:+.4f}'
}))

### 8.1 Train vs Test R² (Overfitting Check)

In [ ]:
overfit_rows = []
for key, res in grid_results.items():
    gs = res['grid_search']
    best_idx = gs.best_index_
    train_r2 = gs.cv_results_['mean_train_score'][best_idx]
    test_r2 = gs.cv_results_['mean_test_score'][best_idx]
    overfit_rows.append({
        'Model': res['model_name'],
        'Dataset': res['dataset_name'],
        'Train R² (CV)': train_r2,
        'Val R² (CV)': test_r2,
        'Gap (Train - Val)': train_r2 - test_r2
    })

overfit_df = pd.DataFrame(overfit_rows)
print('=== Overfitting Check: Train vs Validation R² ===')
print('(A small gap means no significant overfitting)\n')
display(overfit_df.style.format({
    'Train R² (CV)': '{:.4f}',
    'Val R² (CV)': '{:.4f}',
    'Gap (Train - Val)': '{:.4f}'
}).applymap(lambda x: 'background-color: #ffcccc' if isinstance(x, float) and x > 0.02 else '',
            subset=['Gap (Train - Val)']))

---
## 9. Discussion

### 9.1 Key Findings

*(Fill in after running. Template:)*

- **Best performing model:** [Model] on dataset [V?] achieved R² = [?], RMSE = [?] GeV.
- **Feature selection impact:** Reducing V4 from 22 to 8 features [maintained / slightly changed] performance while dramatically reducing training time. The top 3 features (delta_R, sum_pt, sum_E) account for ~80% of importance, confirming that domain-driven feature selection is effective.
- **PCA (V5) performance:** PCA-based dimensionality reduction [improved/reduced] performance compared to V2 because...
- **Scaling insight:** Removing scaling from tree-based pipelines had no effect on their predictions, confirming the professor's feedback.
- **All 4 models comparison:** XGB and RF outperformed KNN and SVR on all datasets, consistent with their ability to model non-linear interactions. SVR was competitive on V4 (fewer features reduce the curse of dimensionality). KNN benefited most from [V4/V5] because...
- **Overfitting:** The Train-Val R² gap is [small/large] for all models, indicating [no/some] overfitting.

### 9.2 Comparison with Literature

| Aspect | Kilic et al. (2023) | Our Study |
|--------|--------------------|-----------|
| Dataset | CERN dielectron (same) | CERN dielectron (same) |
| Target | Z boson mass | Invariant mass M (full range) |
| Best Algorithm | [?] | [?] |
| Features Used | [?] | 8 selected (domain-driven) |
| Dimensionality Reduction | [?] | Feature selection + PCA |
| Best R² | [?] | [?] |
| Best RMSE | [?] | [?] GeV |

### 9.3 Domain-Specific Validation

*(Discuss: Do predictions make physical sense? Does the model correctly predict across all resonance regions (J/ψ, Υ, Z)? Are the most important features (delta_R, sum_E, sum_pt) consistent with the invariant mass formula M = sqrt((E1+E2)² − |p1+p2|²)?)*

### 9.4 Study Limitations

- SVR trained on 30K subsample due to O(n²) complexity, though reduced features helped
- The multimodal target distribution (3 resonance peaks) challenges models optimising MSE
- Engineered features (sum_E) are directly related to the invariant mass formula, so V4's high R² partly reflects learning a known physical relationship
- [Add more based on your findings]

---
## 10. Final Summary

In [ ]:
print('=' * 75)
print('DELIVERABLE 3 – FINAL SUMMARY')
print('=' * 75)
print()
print('Dataset: CERN Electron Collision – 100,000 dielectron events')
print('Task:    Regression – predict invariant mass M (GeV)')
print()
print(f'Models evaluated (screening): KNN, SVR, Random Forest, XGBoost')
print(f'Dataset versions:  V1 (baseline, 16 feat), V2 (log, 16 feat), '
      f'V3 (log target, 16 feat),\n'
      f'                   V4 (selected, 8 feat), V5 (PCA/SVD, ~10 comp)')
print(f'\nModels tuned:      {SELECTED_MODELS}')
print(f'Datasets tuned on: {SELECTED_DATASETS}')
print()
print('--- Best Model (Test Set) ---')
best_row = final_df.loc[final_df['R2'].idxmax()]
print(f'Model:     {best_row["Model"]}')
print(f'Dataset:   {best_row["Dataset"]}')
print(f'R²:        {best_row["R2"]:.4f}')
print(f'RMSE:      {best_row["RMSE"]:.3f} GeV')
print(f'MAE:       {best_row["MAE"]:.3f} GeV')
print(f'MAPE:      {best_row["MAPE"]:.2f}%')
print(f'Params:    {best_row["Best Params"]}')
print()
print('=' * 75)

---
## References

[1] McCauley, T. (2014). Events with two electrons from 2010. CERN Open Data Portal. DOI: 10.7483/OPENDATA.CMS.PCSW.AHVG  
[2] Kilic, H., Ozturk, S., & Yildirim, E. (2023). Machine learning model performances for the Z boson mass identification through dielectron decay channel. *The European Physical Journal Plus*, 138(1), 87.  
[3] Pedregosa, F., et al. (2011). Scikit-learn: Machine Learning in Python. *JMLR*, 12, 2825-2830.  
[4] Chen, T., & Guestrin, C. (2016). XGBoost: A Scalable Tree Boosting System. *KDD 2016*.  
[5] Breiman, L. (2001). Random Forests. *Machine Learning*, 45(1), 5-32.  
[6] Vapnik, V. (1995). *The Nature of Statistical Learning Theory*. Springer.  
[7] Cover, T., & Hart, P. (1967). Nearest neighbor pattern classification. *IEEE Trans. Information Theory*, 13(1), 21-27.  
[8] Jolliffe, I. T. (2002). *Principal Component Analysis* (2nd ed.). Springer. DOI: 10.1007/b98835